### Choose where this notebook runs

This is the **IQM edition**: the states, Bloch spheres and drawings are exact local computations (identical to the IBM edition); the *tomography* of Section 10 and the In-depth analysis — measured one-qubit circuits — are transpiled to the IQM native gate set **PRX + CZ** and run on **Garnet** (20 qubits) or **Emerald** (54 qubits) via IQM Resonance. Pick the backend by editing the single `TARGET`
line below (see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud mock (pipeline check: returns RANDOM bitstrings, no physics) | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |


# The Bloch Sphere — picturing a single qubit

*Code-only notebook; the code comments of the author are in Italian, the guided-reading material (section headers, 🔧 tables, 📐 boxes) in English.*

Every pure state of one qubit can be written, up to an irrelevant global phase, as

$$|\psi\rangle = \cos\frac{\theta}{2}\,|0\rangle + e^{i\varphi}\sin\frac{\theta}{2}\,|1\rangle ,\qquad 0\le\theta\le\pi,\; 0\le\varphi<2\pi ,$$

i.e. as a point $(x, y, z) = (\sin\theta\cos\varphi,\ \sin\theta\sin\varphi,\ \cos\theta)$ on the unit sphere. The notebook:

1. **Setup** — imports.
2. **A generic state** — build $|\psi\rangle$ from $(\theta, \varphi)$ and draw it.
3. **The three bases** — $\{|0\rangle, |1\rangle\}$ (z axis), $\{|+\rangle, |-\rangle\}$ (x axis), $\{|{+i}\rangle, |{-i}\rangle\}$ (y axis).
4. **States prepared by circuits** — H, the general $U(\theta, \varphi, \lambda)$ gate, a sequence of rotations.
5. **LaTeX tools check** (optional).
6. **Comparing and evolving states** — the six cardinal states; an $R_Y$ rotation sweeping a full turn.
7. **Interactive exploration** — sliders for $\theta$ and $\varphi$.
8. **All the representations of a state** — Bloch, LaTeX, text, Q-sphere, Hinton, city.
9. **Putting it together** — circuit, coordinates, amplitudes and probabilities of one state.
10. **Measuring the Bloch vector** — recovering $(x, y, z)$ from measurement counts (state tomography).
11. **🔬 In-depth analysis.**


In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `from iqm_backend import get_backend, run_counts` | the helper module of this folder: `run_counts(circuit, TARGET, shots=...)` = `transpile_to_IQM` (rewrite the circuit with the native `r` = PRX rotations and `cz` gates, routed on the chip's square lattice) → `backend.run` → counts as a plain dict; `run_counts(..., return_transpiled=True)` also returns the transpiled circuit |
| `TARGET = "fake_apollo"` | a local noisy simulator with the 20-qubit Garnet topology: no token, no credits, realistic gate and readout noise; `"garnet:mock"` checks the cloud pipeline but returns RANDOM bitstrings; `"garnet"` / `"emerald"` are the real QPUs (token + credits; IQM Resonance declines circuits above 10 000 native instructions) |
| `get_backend(TARGET)` | the raw Qiskit backend object (name, qubit count, coupling map) if you want to call `backend.run` yourself |

</details>


## 1. Setup


In [ ]:
# Cella 1: Installazione (se necessario)
# !pip install qiskit==2.2.0     # remove the leading '#' to install; '!' runs a shell command from inside a notebook cell
# !pip install qiskit-aer        # Qiskit's high-performance simulator (used in Section 10)
# !pip install matplotlib        # plotting library used by every Qiskit drawer


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `# !pip install ...` | commented out: the course environment already has the packages; uncomment a line only if an import below fails |
| `qiskit==2.2.0` | `==` pins an exact version: the notebook is written for the Qiskit 2.x API |

</details>


In [ ]:
# Cella 2: Import delle librerie
import numpy as np                              # numerics: np.pi, np.cos, complex numbers via 1j (the imaginary unit)
from qiskit import QuantumCircuit               # the object we build circuits with
from qiskit.quantum_info import Statevector     # the exact state vector (amplitudes α, β): no sampling, no noise
import matplotlib.pyplot as plt                 # plotting library behind every Qiskit drawer
from IPython.display import display                      # show a figure (e.g. a circuit drawing) from the middle of a cell
def draw(qc, fold=-1, **kw): return qc.draw("mpl", style="clifford", fold=fold, **kw)   # every circuit picture: matplotlib drawer; fold=-1 = one long row, fold=N wraps after N columns

# Configura matplotlib per Jupyter
# %matplotlib inline   # not needed: inline is the Jupyter default, and with this magic Qiskit's drawers show every figure twice


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `Statevector` | the state of the qubits as a complex vector; for one qubit `[α, β]` with \|α\|² + \|β\|² = 1 |
| `1j` | Python's imaginary unit: `np.exp(1j * phi)` is the phase factor e^{iφ} |
| `draw(qc)` + `display(...)` | our helper around `qc.draw("mpl")`; `display()` shows a figure from the middle of a cell, not only at its end |
| `# %matplotlib inline` | commented out on purpose: inline plotting is already the default; the explicit magic hides it from Qiskit, which then fails to close its figures and every Bloch sphere appeared twice |

</details>


## 2. A generic single-qubit state

The two angles $(\theta, \varphi)$ fix the state: $\theta$ sets the *populations* $P(0) = \cos^2(\theta/2)$, $P(1) = \sin^2(\theta/2)$, and $\varphi$ the *relative phase* between the two amplitudes.


In [ ]:
# Cella 3: Funzione per creare e disegnare uno stato generico
def create_and_draw_state(theta, phi, label=""):   # def name(args): a function; label="" is a default value
    """
    Crea e visualizza uno stato generico sulla sfera di Bloch.

    Parametri:
        theta (float): Angolo polare (0 ≤ θ ≤ π)
        phi (float): Angolo azimutale (0 ≤ φ < 2π)
        label (str): Etichetta per il plot

    Returns:
        Statevector: Lo stato quantistico creato
    """
    # Crea lo stato: |ψ⟩ = cos(θ/2)|0⟩ + e^(iφ)sin(θ/2)|1⟩
    alpha = np.cos(theta / 2)                       # amplitude of |0⟩: real and ≥ 0 (the global phase is chosen so)
    beta = np.exp(1j * phi) * np.sin(theta / 2)     # amplitude of |1⟩: carries the relative phase e^{iφ}
    psi = Statevector([alpha, beta])                # wrap the two numbers as a Qiskit state (checks the normalisation)

    # Disegna sulla sfera di Bloch usando il metodo draw()
    print(f"{label}")                               # f"...{expr}...": an f-string inserts the value of expr
    print(f"θ = {theta:.3f} rad ({np.degrees(theta):.1f}°)")   # :.3f = 3 decimals; np.degrees converts rad -> degrees
    print(f"φ = {phi:.3f} rad ({np.degrees(phi):.1f}°)")
    print(f"|ψ⟩ = ({alpha.real:.3f} + {alpha.imag:.3f}i)|0⟩ + ({beta.real:.3f} + {beta.imag:.3f}i)|1⟩")   # .real / .imag of a complex number
    print(f"P(|0⟩) = {np.abs(alpha)**2:.4f}, P(|1⟩) = {np.abs(beta)**2:.4f}\n")   # Born rule: probability = |amplitude|²

    # Usa il metodo draw() con output 'bloch'
    return psi.draw('bloch')                        # a matplotlib figure: the Bloch vector of psi

# Esempio
display(create_and_draw_state(np.pi/3, np.pi/4, "Stato Generico"))   # θ = 60°, φ = 45°
print("↑ θ = 60° from the north pole, φ = 45° from the x axis: the vector (x, y, z) = (0.612, 0.612, 0.500), closer to |0⟩ than to |1⟩")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `def create_and_draw_state(theta, phi, label="")` | a reusable recipe: angles in, state printed and Bloch figure returned |
| `np.cos(theta / 2)`, `np.sin(theta / 2)` | the *half* angles: θ = π (the south pole) gives cos(π/2) = 0, i.e. the state \|1⟩; orthogonal states are antipodal on the sphere |
| `np.exp(1j * phi)` | the relative phase: it rotates the vector around the z axis and does not change the probabilities |
| `psi.draw('bloch')` | Qiskit's Bloch-sphere plot of the state's vector (x, y, z) |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The state printed above**, $\theta = \pi/3$, $\varphi = \pi/4$:

$$\alpha = \cos\frac{\pi}{6} = \frac{\sqrt3}{2} = 0.866,\qquad \beta = e^{i\pi/4}\sin\frac{\pi}{6} = \frac12\Big(\frac{1}{\sqrt2} + \frac{i}{\sqrt2}\Big) = 0.354 + 0.354\,i .$$

Probabilities: $P(0) = \alpha^2 = 3/4 = 0.7500$ and $P(1) = |\beta|^2 = 1/4 = 0.2500$ (the phase drops out of $|\beta|^2$).

Bloch vector: $x = \sin\theta\cos\varphi = \frac{\sqrt3}{2}\cdot\frac{1}{\sqrt2} = 0.612$, $y = \sin\theta\sin\varphi = 0.612$, $z = \cos\theta = 0.500$. Equivalently $z = P(0) - P(1) = 0.75 - 0.25 = 0.5$ ✓ — the z coordinate *is* the population difference.

</details>


In [ ]:
# Cella 4: Stato generico personalizzabile
# Modifica questi valori
theta = np.pi / 4  # 45 gradi        (polar angle: 0 = |0⟩ at the north pole, π = |1⟩ at the south pole)
phi = np.pi / 6    # 30 gradi        (azimuth: angle from the x axis in the equatorial plane)

# Crea lo stato
alpha = np.cos(theta / 2)                          # amplitude of |0⟩
beta = np.exp(1j * phi) * np.sin(theta / 2)        # amplitude of |1⟩, with the relative phase e^{iφ}
state = Statevector([alpha, beta])

# Disegna sulla sfera di Bloch
print(f"Stato con θ={theta:.3f}, φ={phi:.3f}")
display(state.draw('bloch'))
print(f"↑ θ = 45°, φ = 30°: P(|0⟩) = cos²(22.5°) = {abs(alpha)**2:.3f} — edit theta and phi and re-run the cell")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `theta`, `phi` | the two knobs of the state; edit them and re-run (Shift+Enter) to move the vector |
| `Statevector([alpha, beta])` | the same construction as in Cella 3, written inline |
| `abs(alpha)**2` | `abs` of a complex number is its modulus; `**` is the power operator |

</details>


## 3. The three bases: $Z$, $X$ and $Y$

The six "cardinal" states sit where the coordinate axes cross the sphere. Each pair is an orthonormal basis — the eigenstates of one Pauli matrix — and its two states are **antipodal**.


In [ ]:
# Cella 5: Stati computazionali di base
print("STATI COMPUTAZIONALI DI BASE\n")          # "\n" = new line

# Stato |0⟩
print("Stato |0⟩:")
state_0 = Statevector([1, 0])                    # α = 1, β = 0: θ = 0, the north pole (z = +1)
display(state_0.draw('bloch'))
print("↑ |0⟩: the north pole, Bloch vector (0, 0, +1)")

print("\n" + "="*60 + "\n")                     # "="*60: a string repeated 60 times (a separator line)

# Stato |1⟩
print("Stato |1⟩:")
state_1 = Statevector([0, 1])                    # α = 0, β = 1: θ = π, the south pole (z = −1)
display(state_1.draw('bloch'))
print("↑ |1⟩: the south pole, Bloch vector (0, 0, −1)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `Statevector([1, 0])`, `Statevector([0, 1])` | the computational basis: eigenstates of Z with eigenvalues +1 and −1 |
| `"="*60` | string repetition: `*` on a string concatenates copies |
| north / south pole | \|0⟩ and \|1⟩ are orthogonal (⟨0\|1⟩ = 0) and yet only 180° apart on the sphere: Bloch angles are *half* of Hilbert-space angles |

</details>


In [ ]:
# Cella 6: Stati di superposizione (Hadamard basis)
print("STATI DI SUPERPOSIZIONE\n")

# Stato |+⟩
print("Stato |+⟩ = (|0⟩ + |1⟩)/√2:")
state_plus = Statevector([1/np.sqrt(2), 1/np.sqrt(2)])     # θ = π/2 (equator), φ = 0: the +x axis
display(state_plus.draw('bloch'))
print("↑ |+⟩: on the equator along +x, Bloch vector (+1, 0, 0)")

print("\n" + "="*60 + "\n")

# Stato |-⟩
print("Stato |-⟩ = (|0⟩ - |1⟩)/√2:")
state_minus = Statevector([1/np.sqrt(2), -1/np.sqrt(2)])   # θ = π/2, φ = π (e^{iπ} = −1): the −x axis
display(state_minus.draw('bloch'))
print("↑ |−⟩: on the equator along −x, Bloch vector (−1, 0, 0)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `1/np.sqrt(2)` | equal amplitudes: P(0) = P(1) = 1/2, so z = 0 and the vector lies on the equator |
| the minus sign of \|−⟩ | a relative phase φ = π: same probabilities as \|+⟩, opposite point on the sphere |
| \|±⟩ | the eigenstates of X (the Hadamard basis): H\|0⟩ = \|+⟩, H\|1⟩ = \|−⟩ |

</details>


In [ ]:
# Cella 7: Stati circolari (Y basis)
print("STATI CIRCOLARI\n")

# Stato |+i⟩
print("Stato |+i⟩ = (|0⟩ + i|1⟩)/√2:")
state_plusi = Statevector([1/np.sqrt(2), 1j/np.sqrt(2)])   # relative phase i = e^{iπ/2}: φ = π/2, the +y axis
display(state_plusi.draw('bloch'))
print("↑ |+i⟩: on the equator along +y, Bloch vector (0, +1, 0)")

print("\n" + "="*60 + "\n")

# Stato |-i⟩
print("Stato |-i⟩ = (|0⟩ - i|1⟩)/√2:")
state_minusi = Statevector([1/np.sqrt(2), -1j/np.sqrt(2)]) # relative phase −i = e^{i3π/2}: φ = 3π/2, the −y axis
display(state_minusi.draw('bloch'))
print("↑ |−i⟩: on the equator along −y, Bloch vector (0, −1, 0)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `1j/np.sqrt(2)` | an imaginary amplitude: the relative phase i = e^{iπ/2} puts the vector on the +y axis |
| \|±i⟩ | the eigenstates of Y; measured in the Z basis they give 50/50 like \|±⟩ — only the phase distinguishes the four equatorial states |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**Where the six cardinal states sit.** Match each state with $\cos\frac{\theta}{2}|0\rangle + e^{i\varphi}\sin\frac{\theta}{2}|1\rangle$:

| state | $\theta$ | $\varphi$ | $(x, y, z)$ |
|---|---|---|---|
| $\vert 0\rangle$ | $0$ | any | $(0, 0, 1)$ |
| $\vert 1\rangle$ | $\pi$ | any | $(0, 0, -1)$ |
| $\vert {+}\rangle = (\vert 0\rangle + \vert 1\rangle)/\sqrt2$ | $\pi/2$ | $0$ | $(1, 0, 0)$ |
| $\vert {-}\rangle = (\vert 0\rangle - \vert 1\rangle)/\sqrt2$ | $\pi/2$ | $\pi$ | $(-1, 0, 0)$ |
| $\vert {+i}\rangle = (\vert 0\rangle + i\vert 1\rangle)/\sqrt2$ | $\pi/2$ | $\pi/2$ | $(0, 1, 0)$ |
| $\vert {-i}\rangle = (\vert 0\rangle - i\vert 1\rangle)/\sqrt2$ | $\pi/2$ | $3\pi/2$ | $(0, -1, 0)$ |

**Overlaps.** $|\langle +|{+i}\rangle|^2 = \big|\tfrac12(1 + i)\big|^2 = \tfrac12$: states at 90° on the sphere have overlap $\tfrac12$; antipodal states (180°) have overlap 0. In general $|\langle\psi|\chi\rangle|^2 = \tfrac12(1 + \vec r_\psi\cdot\vec r_\chi)$.

</details>


## 4. States prepared by circuits

On a quantum computer we never write amplitudes by hand: we start from $|0\rangle$ and apply gates. `Statevector(qc)` computes the state the circuit `qc` prepares from $|0\rangle$.


In [ ]:
# Cella 8: Creare stati usando circuiti quantistici e disegnare
print("STATI CREATI CON CIRCUITI QUANTISTICI\n")

# Stato |+⟩ usando Hadamard
qc_plus = QuantumCircuit(1)          # QuantumCircuit(number_of_qubits): starts in |0⟩
qc_plus.h(0)                         # Hadamard on qubit 0: |0⟩ -> |+⟩
print("Circuito per |+⟩:")
display(draw(qc_plus)); print("↑ one qubit, one Hadamard gate")

# Ottieni e disegna lo stato
state_plus = Statevector(qc_plus)    # simulate the circuit exactly: the state it prepares from |0⟩
print("\nStato risultante sulla sfera di Bloch:")
display(state_plus.draw('bloch'))
print("↑ H|0⟩ = |+⟩: the vector moved from the north pole to +x (a half turn around the (x+z)/√2 axis)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumCircuit(1)` | a one-qubit circuit; every qubit starts in \|0⟩ |
| `qc_plus.h(0)` | append a Hadamard to qubit 0; nothing is executed yet, the circuit is only a list of instructions |
| `Statevector(qc_plus)` | exact simulation: multiply the gate matrices onto \|0⟩ (feasible only for few qubits — 2ⁿ amplitudes) |

</details>


In [ ]:
# Cella 9: Rotazioni generiche usando circuiti
def create_state_with_circuit(theta, phi, lambda_param=0):
    """
    Crea uno stato usando una porta U generico.

    Parametri:
        theta, phi, lambda_param: Parametri della porta U
    """
    qc = QuantumCircuit(1)
    qc.u(theta, phi, lambda_param, 0)     # the general one-qubit gate U(θ, φ, λ) on qubit 0 (last argument = the qubit)

    print("Circuito:")
    display(draw(qc)); print("↑ a single U(θ, φ, λ) gate: every one-qubit unitary is one of these (up to a global phase)")

    # Ottieni lo stato e disegnalo
    state = Statevector(qc)
    print(f"\nStato risultante:")
    print(f"Coefficienti: {state.data}")    # .data = the numpy array of amplitudes [α, β]

    return state.draw('bloch')

# Esempio
print("Rotazione generica con U(π/3, π/4, 0):\n")
display(create_state_with_circuit(np.pi/3, np.pi/4, 0))
print("↑ U(π/3, π/4, 0)|0⟩ is the state of Cella 3: same θ and φ, same point on the sphere")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.u(theta, phi, lambda_param, 0)` | U(θ, φ, λ) = [[cos(θ/2), −e^{iλ} sin(θ/2)], [e^{iφ} sin(θ/2), e^{i(φ+λ)} cos(θ/2)]] |
| `lambda_param` | λ multiplies only the \|1⟩ column, so it does not change U\|0⟩: it matters when U acts on other states |
| `state.data` | the raw complex amplitudes; here [0.866, 0.354+0.354j], exactly α and β of Cella 3 |

</details>


In [ ]:
# Cella 10: Sequenza di rotazioni
print("SEQUENZA DI ROTAZIONI\n")

# Crea circuito con rotazioni successive
qc = QuantumCircuit(1)
qc.ry(np.pi/4, 0)  # Rotazione attorno Y      (tilts the vector 45° from the north pole towards +x)
qc.rz(np.pi/3, 0)  # Rotazione attorno Z      (then turns it 60° around the z axis)

print("Circuito:")
display(draw(qc)); print("↑ RY(π/4) followed by RZ(π/3): gates act left to right in time")

# Visualizza stato finale
state = Statevector(qc)
print("\nStato finale sulla sfera di Bloch:")
display(state.draw('bloch'))
print("↑ the final vector: θ = 45°, φ = 60°")

# Mostra anche rappresentazione testuale
print("\nRappresentazione LaTeX dello stato:")
display(state.draw('latex'))            # rendered by the notebook (MathJax): no LaTeX installation needed


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.ry(angle, q)`, `qc.rz(angle, q)` | rotations of the Bloch vector by `angle` around the y and z axes: R_a(α) = exp(−iα σ_a/2) |
| order of the calls | the first call is the first gate in time; in matrix form the state is RZ·RY\|0⟩ (right to left) |
| `state.draw('latex')` | the state as a formula; note the common factor e^{−iπ/6} (a global phase, invisible on the sphere) |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The two rotations by hand.** $R_Y(\alpha) = \begin{pmatrix}\cos\frac\alpha2 & -\sin\frac\alpha2\\ \sin\frac\alpha2 & \cos\frac\alpha2\end{pmatrix}$ and $R_Z(\beta) = \begin{pmatrix}e^{-i\beta/2} & 0\\ 0 & e^{i\beta/2}\end{pmatrix}$, so

$$R_Z\!\left(\tfrac\pi3\right)R_Y\!\left(\tfrac\pi4\right)|0\rangle = e^{-i\pi/6}\cos\tfrac\pi8\,|0\rangle + e^{i\pi/6}\sin\tfrac\pi8\,|1\rangle = e^{-i\pi/6}\Big(\cos\tfrac\pi8\,|0\rangle + e^{i\pi/3}\sin\tfrac\pi8\,|1\rangle\Big).$$

Inside the brackets is the standard form with $\theta = \pi/4$ (45°) and $\varphi = \pi/3$ (60°); the factor $e^{-i\pi/6}$ is a global phase. Numerically $\cos\frac\pi8 = 0.924$, $\sin\frac\pi8 = 0.383$, so the amplitudes printed by `draw('latex')` are $0.924\,e^{-i\pi/6} = 0.800 - 0.462\,i$ and $0.383\,e^{i\pi/6} = 0.331 + 0.191\,i$.

</details>


## 5. Checking the LaTeX tools (optional)

Some Qiskit drawers (`qc.draw("latex")`, not used here) need a LaTeX installation. The two cells below only *look* for the programs `pdflatex` and `dvipng`; a missing program is reported, not an error.


In [ ]:
import subprocess                    # run an external program and capture what it prints
import shutil                        # shutil.which(name): full path of a program on the PATH, or None if absent
if shutil.which('pdflatex'):         # guard: subprocess.run raises FileNotFoundError when the program does not exist
    result = subprocess.run(['pdflatex', '--version'], capture_output=True, text=True)   # capture_output: keep stdout/stderr; text=True: as str, not bytes
    print(result.stdout if result.returncode == 0 else "pdflatex NON trovato")             # conditional expression; returncode 0 = success
else:
    print("pdflatex NON trovato")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `shutil.which('pdflatex')` | the guard we added: without it, a computer without LaTeX stopped the notebook here with FileNotFoundError |
| `subprocess.run([...], capture_output=True, text=True)` | runs `pdflatex --version` as if typed in a terminal; the list is program + arguments |
| `a if cond else b` | Python's conditional expression (the ternary operator) |

</details>


In [ ]:
if shutil.which('dvipng'):           # same guard as above
    result = subprocess.run(['dvipng', '--version'], capture_output=True, text=True)
    print(result.stdout if result.returncode == 0 else "dvipng NON trovato")
else:
    print("dvipng NON trovato")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `dvipng` | converts LaTeX output to PNG images; needed together with pdflatex by `qc.draw("latex")` |
| `result.stdout` | the text the program printed (its version banner) |

</details>


## 6. Comparing and evolving states

First the six cardinal states side by side; then an $R_Y$ rotation that sweeps the vector through a full turn in the $xz$ plane.


In [ ]:
# Cella 11: Confronto visivo di più stati usando draw()
def compare_states_on_bloch(states_dict):
    """
    Confronta visivamente più stati usando il metodo draw().

    Parametri:
        states_dict: Dizionario {label: Statevector}
    """
    print("CONFRONTO STATI SULLA SFERA DI BLOCH")
    print("="*70)

    for label, state in states_dict.items():          # .items() yields (key, value) pairs of a dict
        print(f"\n{label}:")
        print(f"Coefficienti: α = {state.data[0]:.3f}, β = {state.data[1]:.3f}")   # :.3f also works on complex numbers
        display(state.draw('bloch'))
        print(f"↑ {label} on the Bloch sphere")
        print("-"*70)

# Crea diversi stati
states = {                                            # a dict {label: state}; insertion order is kept
    '|0⟩': Statevector([1, 0]),
    '|1⟩': Statevector([0, 1]),
    '|+⟩': Statevector([1/np.sqrt(2), 1/np.sqrt(2)]),
    '|-⟩': Statevector([1/np.sqrt(2), -1/np.sqrt(2)]),
    '|+i⟩': Statevector([1/np.sqrt(2), 1j/np.sqrt(2)]),
    '|-i⟩': Statevector([1/np.sqrt(2), -1j/np.sqrt(2)])
}

compare_states_on_bloch(states)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `states = {...}` | a dictionary label → state: the six cardinal states of Section 3 |
| `for label, state in states_dict.items()` | loop over the pairs; tuple unpacking assigns key and value in one line |
| `state.data[0]` | indexing the amplitude array: [0] is α (the \|0⟩ amplitude), [1] is β |

</details>


In [ ]:
# Cella 12: Evoluzione di uno stato sulla sfera di Bloch (VERSIONE CORRETTA)
print("EVOLUZIONE TEMPORALE DI UNO STATO")
print("="*70)
print("Rotazione RY da 0 a 2π attorno all'asse Y\n")

# Stato iniziale
print("Stato iniziale (angle=0):")
state_0 = Statevector(QuantumCircuit(1))       # an empty circuit prepares |0⟩
display(state_0.draw('bloch'))
print("↑ start: |0⟩ at the north pole")

# Applica rotazioni progressive
angles = [np.pi/4, np.pi/2, 3*np.pi/4, np.pi, 5*np.pi/4, 3*np.pi/2, 7*np.pi/4, 2*np.pi]   # steps of 45°

for i, angle in enumerate(angles, 1):          # enumerate(seq, 1): pairs (counter starting at 1, element)
    qc = QuantumCircuit(1)
    qc.ry(angle, 0)                            # a fresh circuit for each angle: RY(angle)|0⟩
    state = Statevector(qc)

    print(f"\nStep {i}: RY({angle:.3f} rad = {np.degrees(angle):.0f}°)")
    print(f"Coefficienti: α = {state.data[0]:.3f}, β = {state.data[1]:.3f}")
    display(state.draw('bloch'))
    print(f"↑ after RY({np.degrees(angle):.0f}°): the vector has turned {np.degrees(angle):.0f}° in the xz plane")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `enumerate(angles, 1)` | counts the loop steps from 1 while walking through the list |
| `qc.ry(angle, 0)` | a rotation of the Bloch vector by `angle` around y: from \|0⟩ it moves along the xz great circle |
| the printed coefficients | α = cos(angle/2), β = sin(angle/2): they are periodic in 4π, the picture in 2π (see the box) |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**A full turn is not the identity.** $R_Y(\alpha)|0\rangle = \cos\frac\alpha2|0\rangle + \sin\frac\alpha2|1\rangle$. Check the steps:

* $\alpha = \pi/2$: $\big(\cos\frac\pi4, \sin\frac\pi4\big) = (0.707, 0.707)$ — the state $|+\rangle$ on the equator;
* $\alpha = \pi$: $(\cos\frac\pi2, \sin\frac\pi2) = (0, 1)$ — the state $|1\rangle$ at the south pole;
* $\alpha = 2\pi$: $(\cos\pi, \sin\pi) = (-1, 0)$, i.e. $R_Y(2\pi)|0\rangle = -|0\rangle$.

The last step prints $\alpha = -1.000$: the vector is back at the north pole (a global phase $-1$ is invisible on the sphere), but the amplitude changed sign. Spin-½ rotations have period $4\pi$; the sign becomes observable when the rotation is *controlled* (it turns into a relative phase — the mechanism behind phase kickback).

</details>


## 7. Interactive exploration

Move the sliders to change $\theta$ and $\varphi$; the state and its Bloch vector are recomputed at every move. (The sliders need a live Jupyter kernel: in a static copy of the notebook only the widget placeholder is shown.)


In [ ]:
# Cella 13: Stato generico con slider interattivo
from ipywidgets import interact, FloatSlider   # Jupyter widgets: interactive controls bound to a Python function

@interact(                                     # decorator: builds one slider per argument and re-calls the function at every move
    theta=FloatSlider(value=np.pi/4, min=0, max=np.pi, step=0.05,
                     description='θ (rad):'),
    phi=FloatSlider(value=np.pi/4, min=0, max=2*np.pi, step=0.05,
                   description='φ (rad):')
)
def interactive_state(theta, phi):
    """Widget interattivo per esplorare la sfera di Bloch."""
    alpha = np.cos(theta / 2)
    beta = np.exp(1j * phi) * np.sin(theta / 2)
    state = Statevector([alpha, beta])

    print(f"θ = {theta:.3f} rad ({np.degrees(theta):.1f}°)")
    print(f"φ = {phi:.3f} rad ({np.degrees(phi):.1f}°)")
    print(f"|ψ⟩ = ({alpha.real:.3f}{alpha.imag:+.3f}i)|0⟩ + ({beta.real:.3f}{beta.imag:+.3f}i)|1⟩")   # :+.3f always prints the sign

    display(state.draw('bloch'))
    print("↑ the state chosen with the sliders")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `@interact(theta=FloatSlider(...), phi=FloatSlider(...))` | a decorator (the `@` line modifies the function below it): one slider per argument, the function re-runs whenever a slider moves |
| `FloatSlider(value, min, max, step, description)` | a slider for a float; θ ranges over [0, π] and φ over [0, 2π] — the whole sphere |
| `{alpha.imag:+.3f}` | format spec `+`: always show the sign, so the output reads 0.354+0.354i |
| static copies | widgets need a running kernel: in an exported or headless-executed notebook the cell shows only the widget placeholder |

</details>


## 8. All the representations of a state

The same state $\theta = \pi/3$, $\varphi = \pi/4$ (the state of Section 2) drawn in every format `Statevector.draw` offers.


In [ ]:
# Cella 14: Tutte le rappresentazioni disponibili di uno stato
print("DIVERSE RAPPRESENTAZIONI DI UNO STATO\n")

# Crea uno stato generico
theta, phi = np.pi/3, np.pi/4                   # tuple assignment: two variables in one line
alpha = np.cos(theta / 2)
beta = np.exp(1j * phi) * np.sin(theta / 2)
state = Statevector([alpha, beta])

print("1. Sfera di Bloch:")
display(state.draw('bloch'))
print("↑ the Bloch vector (0.612, 0.612, 0.500)")

print("\n2. Notazione LaTeX:")
display(state.draw('latex'))                    # the ket written as a formula

print("\n3. Rappresentazione testuale:")
display(state.draw('text'))                     # plain-text array of amplitudes

print("\n4. Q-sphere:")
display(state.draw('qsphere'))
print("↑ Q-sphere: one dot per basis state; dot size = probability, colour = phase (|1⟩ has phase π/4)")

print("\n5. Hinton plot:")
display(state.draw('hinton'))
print("↑ Hinton plot of the density matrix ρ = |ψ⟩⟨ψ|: square size = magnitude of each entry, real and imaginary parts separately")

print("\n6. City plot:")
display(state.draw('city'))
print("↑ city plot: the same density matrix as 3-D bars (real part left, imaginary part right)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `state.draw('latex')`, `'text'` | the state as a formula / as a plain array of amplitudes |
| `state.draw('qsphere')` | the Q-sphere: also works for many qubits (one point per basis state), unlike the Bloch sphere |
| `state.draw('hinton')`, `'city'` | two pictures of the density matrix ρ = \|ψ⟩⟨ψ\|; ρ₀₀ = 0.75, ρ₁₁ = 0.25, ρ₀₁ = αβ* carries the phase |

</details>


## 9. Putting it together

One function that shows, for a given $(\theta, \varphi)$, the preparing circuit, the Bloch coordinates, the amplitudes and the probabilities.


In [ ]:
# Cella 15: Funzione completa per esplorare stati
def explore_bloch_state(theta, phi):
    """
    Funzione completa per esplorare uno stato sulla sfera di Bloch.
    Mostra circuito, stato e visualizzazioni multiple.
    """
    # Crea circuito
    qc = QuantumCircuit(1)
    qc.u(theta, phi, 0, 0)                      # U(θ, φ, 0)|0⟩ = cos(θ/2)|0⟩ + e^{iφ} sin(θ/2)|1⟩

    print("="*70)
    print(f"ESPLORAZIONE STATO: θ={theta:.3f} rad, φ={phi:.3f} rad")
    print("="*70)

    # Mostra circuito
    print("\nCircuito quantistico:")
    display(draw(qc)); print("↑ the preparing circuit: one U gate on |0⟩")

    # Ottieni stato
    state = Statevector(qc)

    # Coordinate Bloch
    x = np.sin(theta) * np.cos(phi)             # spherical -> Cartesian coordinates of the Bloch vector
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)

    print(f"\nCoordinate sulla sfera di Bloch:")
    print(f"  x = {x:.4f}")
    print(f"  y = {y:.4f}")
    print(f"  z = {z:.4f}")

    # Coefficienti
    print(f"\nCoefficienti dello stato:")
    print(f"  α = {state.data[0]:.4f}")
    print(f"  β = {state.data[1]:.4f}")

    print(f"\nProbabilità:")
    print(f"  P(|0⟩) = {np.abs(state.data[0])**2:.4f}")
    print(f"  P(|1⟩) = {np.abs(state.data[1])**2:.4f}")

    # Visualizza sulla sfera di Bloch
    print("\nVisualizzazione sulla sfera di Bloch:")
    return state.draw('bloch')

# Esempio
display(explore_bloch_state(np.pi/3, np.pi/4))
print("↑ the same state as in Sections 2 and 8: (x, y, z) = (0.6124, 0.6124, 0.5000)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `x = sin θ cos φ`, `y = sin θ sin φ`, `z = cos θ` | the Bloch vector in Cartesian coordinates; these are also the expectation values ⟨X⟩, ⟨Y⟩, ⟨Z⟩ of the state |
| `np.abs(state.data[0])**2` | the Born rule on the simulated amplitudes: 0.7500 and 0.2500, as computed in Section 2 |
| `return state.draw('bloch')` | the figure is returned and shown by `display(...)` in the calling line |

</details>


## 10. Measuring the Bloch vector

A real device gives no amplitudes, only measurement **counts**. Yet the Bloch vector can be measured: $z = \langle Z\rangle = P(0) - P(1)$ in the computational basis; for $x$ and $y$ we first rotate the $X$ or $Y$ axis onto $Z$ and then measure. This is single-qubit **state tomography**. The first cell chooses where the circuits run; the second does the tomography of the state of Section 2.


In [ ]:
backend = get_backend(TARGET)                               # the IQM backend chosen by the TARGET line at the top
print(f"{backend.name}: {backend.num_qubits} qubits, {len(backend.coupling_map.get_edges())} couplers")   # coupling map = qubit pairs that can do a CZ

def run_circuit(circuit, shots=4096):
    """Transpile to the IQM native gates + run on TARGET + counts in one call (used by the tomography and the In-depth analysis)."""
    return run_counts(circuit, backend, shots=shots)         # run_counts accepts a target name or a backend object


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `get_backend(TARGET)` | the Qiskit backend object chosen by the `TARGET` line at the top: `fake_apollo` = local noisy 20-qubit model (no token); `garnet:mock` = cloud pipeline check (random bitstrings); `garnet` / `emerald` = the real QPUs |
| `backend.coupling_map.get_edges()` | the pairs of qubits joined by a coupler: a two-qubit gate between qubits that are not neighbours on the square lattice needs extra SWAPs |
| `run_counts(circuit, backend, shots=...)` | `transpile_to_IQM` (rewrite in the native `r` = PRX rotations and `cz`, routed on the lattice) → `backend.run` → counts as a plain dict {bitstring: shots}; IQM Resonance declines circuits above 10 000 native instructions |
| `run_circuit(circuit, shots)` | the same name as in the IBM edition, so the tomography and the In-depth cells are identical in both editions |

</details>


In [ ]:
def bloch_from_counts(theta, phi, shots=4096):
    """Estimate (x, y, z) of U(θ, φ, 0)|0⟩ from counts in the X, Y and Z bases."""
    est = {}
    for axis in "xyz":                         # iterating over a string yields its characters
        qc = QuantumCircuit(1, 1)              # QuantumCircuit(qubits, classical_bits)
        qc.u(theta, phi, 0, 0)                 # prepare the state
        if axis == "x":
            qc.h(0)                            # H maps the X eigenstates |±⟩ onto |0⟩/|1⟩
        if axis == "y":
            qc.sdg(0); qc.h(0)                 # S† then H maps the Y eigenstates |±i⟩ onto |0⟩/|1⟩
        qc.measure(0, 0)                       # measure(qubit, classical_bit)
        counts = run_circuit(qc, shots)
        est[axis] = (counts.get("0", 0) - counts.get("1", 0)) / shots   # ⟨σ⟩ = P(0) − P(1); dict.get(key, 0) = 0 if the key is absent
    return est, qc                             # also return the last circuit (the y measurement) to draw it

theta, phi = np.pi/3, np.pi/4
est, qc_y = bloch_from_counts(theta, phi)
display(draw(qc_y)); print("↑ the y-axis measurement: prepare with U, rotate the y axis onto z with S† and H, then measure")
exact = (np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta))
for (axis, value), ex in zip(est.items(), exact):   # zip pairs the estimated and the exact components
    print(f"  {axis}: measured {value:+.3f}   exact {ex:+.3f}")
print(f"statistical error per component ≈ 1/√shots = {1/np.sqrt(4096):.3f}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.h(0)` before `measure` | H swaps the X and Z axes, so P(0) − P(1) after it is ⟨X⟩ of the state before it |
| `qc.sdg(0); qc.h(0)` | S† turns the Y axis onto X, then H turns X onto Z: the pair measures ⟨Y⟩ |
| `counts.get("0", 0)` | the number of shots that gave 0 (0 if the outcome never occurred) |
| `zip(est.items(), exact)` | walk two sequences in step: estimated and exact components side by side |
| measured vs exact | the estimates scatter by about 1/√4096 ≈ 0.016 (shot noise); on a noisy device they are also shrunk towards 0 by decoherence and readout errors |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**Why the rotations work.** $H$ exchanges the $X$ and $Z$ axes ($HXH = Z$), so measuring $Z$ after $H$ gives
$$P(0) - P(1) = \langle\psi|H Z H|\psi\rangle = \langle\psi|X|\psi\rangle = x .$$
For $y$ we need a unitary $V$ with $V^\dagger Z V = Y$: $V = H S^\dagger$ does it, since $S X S^\dagger = Y$ gives $S H Z H S^\dagger = Y$.

**The numbers.** For $\theta = \pi/3$, $\varphi = \pi/4$: $(x, y, z) = (0.612, 0.612, 0.500)$, i.e. $P(0) = (1 + 0.612)/2 = 0.806$ in the X and Y measurements and $0.750$ in the Z measurement. Each estimate is an average of $\pm1$ outcomes with variance $1 - r^2$, so its standard deviation is $\sqrt{(1 - r^2)/N} \le 1/\sqrt{4096} = 0.016$: on the ideal simulator the measured values agree with the exact ones to about two decimals; on a noisy device (the IQM edition) all three are also shrunk towards 0 by decoherence and readout errors.

</details>


## 11. 🔬 In-depth analysis

1. **Expectation values.** For the state with $\theta = 2\pi/3$, $\varphi = 5\pi/4$ compute $\langle X\rangle$, $\langle Y\rangle$, $\langle Z\rangle$ exactly with `Statevector.expectation_value(Pauli("X"))` (and Y, Z), and check them against $(\sin\theta\cos\varphi, \sin\theta\sin\varphi, \cos\theta)$.
2. **Antipodes are orthogonal.** For the same state, build the state at the antipodal point ($\theta' = \pi - \theta$, $\varphi' = \varphi + \pi$) and verify that its overlap $|\langle\psi|\psi'\rangle|^2$ is 0; then check the formula $|\langle\psi|\chi\rangle|^2 = \frac12(1 + \vec r_\psi\cdot\vec r_\chi)$ for $\chi = |{+i}\rangle$.
3. **Tomography of $|{+i}\rangle$.** Use `bloch_from_counts` to measure the Bloch vector of $|{+i}\rangle$ ($\theta = \pi/2$, $\varphi = \pi/2$) and compute the length $|\vec r|$ of the estimated vector. A pure state has $|\vec r| = 1$: how far from 1 is your estimate, and why?

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [ ]:
# 🔬 In-depth analysis — your code here



In [ ]:
# Reference solution
from qiskit.quantum_info import Pauli                 # Pauli("X"), Pauli("Y"), Pauli("Z") as operators

def ket(theta, phi):                                  # the standard parametrisation as a Statevector
    return Statevector([np.cos(theta/2), np.exp(1j*phi)*np.sin(theta/2)])
def bloch(theta, phi):                                # its Bloch vector as a numpy array
    return np.array([np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)])

# (1) expectation values = Bloch coordinates
th, ph = 2*np.pi/3, 5*np.pi/4
psi = ket(th, ph)
xyz = [psi.expectation_value(Pauli(p)).real for p in "XYZ"]   # ⟨ψ|σ|ψ⟩ is real for a Hermitian σ; .real drops the 0j
print(f"(1) <X>,<Y>,<Z> = {np.round(xyz, 4)}   formula = {np.round(bloch(th, ph), 4)}")

# (2) the antipode and the overlap formula
anti = ket(np.pi - th, ph + np.pi)
print(f"(2) |<psi|antipode>|^2 = {abs(psi.inner(anti))**2:.2e}")    # .inner(other) = ⟨self|other⟩
chi = Statevector([1/np.sqrt(2), 1j/np.sqrt(2)])                   # |+i⟩, Bloch vector (0, 1, 0)
lhs = abs(psi.inner(chi))**2
rhs = 0.5 * (1 + bloch(th, ph) @ np.array([0, 1, 0]))              # @ = dot product of numpy arrays
print(f"    |<psi|+i>|^2 = {lhs:.4f}   (1 + r_psi·r_chi)/2 = {rhs:.4f}")

# (3) tomography of |+i⟩ from counts
est_pi, _ = bloch_from_counts(np.pi/2, np.pi/2)                    # _ = a value we do not need (the circuit)
r = np.array([est_pi[a] for a in "xyz"])
print(f"(3) measured r = {np.round(r, 3)}, |r| = {np.linalg.norm(r):.3f}  (pure state: 1; shot noise and device noise make it differ)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `psi.expectation_value(Pauli("X"))` | ⟨ψ\|X\|ψ⟩ computed exactly from the amplitudes: the Bloch coordinates are the three Pauli expectation values |
| `ket(np.pi - th, ph + np.pi)` | the antipodal point: r → −r, and antipodal states are orthogonal (overlap ~1e-32, i.e. 0 up to rounding) |
| `psi.inner(chi)` | the inner product ⟨ψ\|χ⟩; its squared modulus is the overlap (the probability of finding χ when ψ is prepared) |
| `np.linalg.norm(r)` | the length of the estimated vector; values slightly below or above 1 come from shot noise, and on a noisy device decoherence and readout errors shorten it |

</details>
